# 06 — Co-evolução Threshold + Hiperparâmetros | Penalidade Suave

## Motivação

Os notebooks 05/02/03/04 revelaram um problema estrutural no fitness:

```python
# Cliff — o problema:
return f2 if recall >= 0.80 else recall + FITNESS_PENALTY
```

Com threshold 0.50, o recall do baseline no CV é ~0.515 — toda a população fica abaixo
do piso, otimizando recall puro em vez de F2. O AG nunca cruza a descontinuidade.

## Solução dupla implementada aqui

**1. Penalidade quadrática suave** — elimina a falésia, mantém pressão seletiva contínua.
Biologicamente: fitness contínuo como qualquer traço quantitativo real (altura, fertilidade).

```python
return f2 - ALPHA * max(0.0, 0.80 - recall) ** 2
```

**2. Co-evolução do threshold** — o limiar de decisão vira o 8º gene.
O AG aprende simultaneamente *quais hiperparâmetros usar* e *onde cortar a probabilidade*.
Biologicamente: coevolução hospedeiro–parasita — dois traços co-adaptam sob a mesma pressão.

**3. Mutação adaptativa** — quando a população estagna por K gerações, a taxa de mutação
é amplificada automaticamente. Biologicamente: mutagênese induzida por estresse.

**4. Cenários A/B/C** — o melhor config é rodado nos três conjuntos de features para
responder se o AG se comporta diferente com outros cenários de dados.

In [ ]:
import gc
import json
import math
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from joblib import Parallel, delayed

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    fbeta_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.utils.class_weight import compute_sample_weight
from utils.experiment_utils import apply_scenario

In [ ]:
warnings.filterwarnings("ignore")

RANDOM_STATE         = 42
CV_SPLITS            = 3
MIN_RECALL_THRESHOLD = 0.80

# Coeficiente da penalidade quadrática suave
# Ao recall=0.70 → penalidade=2.0*(0.10)^2=0.02 | ao recall=0.50 → 0.18 | ao recall=0.00 → 1.28
ALPHA = 2.0

# Mutação adaptativa: após STAGNATION_WINDOW gerações sem melhora, multiplica por MUTATION_BOOST
STAGNATION_WINDOW = 3
MUTATION_BOOST    = 2.0

DATA_DIR      = Path("../data")
METRICS_DIR   = Path("../results/metrics")
ARTIFACTS_DIR = Path("../results/artifacts")
FIGURES_DIR   = Path("../results/figures")

for d in [METRICS_DIR, ARTIFACTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

N_JOBS = max(1, (os.cpu_count() - 4) // 2)
print(f"CPUs disponíveis : {os.cpu_count()}")
print(f"N_JOBS           : {N_JOBS}")

In [ ]:
SCENARIO_DESCRIPTIONS = {
    "A": "dummies + bools",
    "B": "ordinal + contínuo",
    "C": "todos",
}

def load_scenario_data(scenario: str) -> tuple:
    X_train_full = pd.read_parquet(DATA_DIR / "X_train.parquet")
    X_test_full  = pd.read_parquet(DATA_DIR / "X_test.parquet")
    y_train = pd.read_parquet(DATA_DIR / "y_train.parquet").iloc[:, 0]
    y_test  = pd.read_parquet(DATA_DIR / "y_test.parquet").iloc[:, 0]
    X_train = apply_scenario(X_train_full, scenario)
    X_test  = apply_scenario(X_test_full,  scenario)
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = load_scenario_data("B")
print(f"Cenário B ({SCENARIO_DESCRIPTIONS['B']})  |  X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print("Distribuição y_train:", y_train.value_counts(normalize=True).round(3).to_dict())

## Baseline (RandomizedSearch Fase 1)

Referência fixa para avaliar se o AG co-evolutivo traz ganho real.

In [ ]:
METRIC_NAMES = ["recall", "f2", "precision", "roc_auc", "average_precision", "brier"]

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "threshold":         float(threshold),
        "recall":            recall_score(y_true, y_pred, zero_division=0),
        "f2":                fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "f1":                f1_score(y_true, y_pred, zero_division=0),
        "precision":         precision_score(y_true, y_pred, zero_division=0),
        "roc_auc":           roc_auc_score(y_true, y_prob),
        "average_precision": average_precision_score(y_true, y_prob),
        "brier":             brier_score_loss(y_true, y_prob),
    }

def evaluate_on_test(params: dict, threshold: float, X_te=None, X_tr=None, y_tr=None, y_te=None) -> dict:
    """Treina com sample_weight, avalia no teste com o threshold indicado."""
    X_tr = X_tr if X_tr is not None else X_train
    y_tr = y_tr if y_tr is not None else y_train
    X_te = X_te if X_te is not None else X_test
    y_te = y_te if y_te is not None else y_test
    sw    = compute_sample_weight(class_weight="balanced", y=y_tr)
    model = HistGradientBoostingClassifier(**params, random_state=RANDOM_STATE)
    model.fit(X_tr, y_tr, sample_weight=sw)
    y_prob = model.predict_proba(X_te)[:, 1]
    return compute_metrics(y_te, y_prob, threshold=threshold)

def load_baseline_params(metrics_dir: Path) -> dict:
    operational = json.loads((metrics_dir / "best_model_operational_metrics.json").read_text())
    return operational["best_params"]

BASELINE_PARAMS = load_baseline_params(METRICS_DIR)

baseline_050 = evaluate_on_test(BASELINE_PARAMS, threshold=0.50)
baseline_040 = evaluate_on_test(BASELINE_PARAMS, threshold=0.40)

baseline_summary = pd.DataFrame([
    {"threshold": "0.50 (padrão)",      **{m: baseline_050[m] for m in METRIC_NAMES}},
    {"threshold": "0.40 (operacional)", **{m: baseline_040[m] for m in METRIC_NAMES}},
]).set_index("threshold")

print("=== BASELINE — HistGB (RandomizedSearch Fase 1, cenário B) ===")
display(baseline_summary.round(4))
print(f"\nMeta: F2 > {baseline_040['f2']:.4f} com recall >= {MIN_RECALL_THRESHOLD}")

gc.collect()

## Co-evolução — Genoma e Fitness

### Representação genética (8 genes)

| Gene | Parâmetro | Valores | Qtd |
|---|---|---|---|
| 1 | `max_iter` | 100..500 | 8 |
| 2 | `learning_rate` | 0.01..0.20 | 7 |
| 3 | `max_leaf_nodes` | 15..127 | 7 |
| 4 | `min_samples_leaf` | 10..150 | 7 |
| 5 | `l2_regularization` | 0.0..5.0 | 7 |
| 6 | `max_depth` | None, 3..15 | 6 |
| 7 | `max_bins` | 63..255 | 3 |
| **8** | **`threshold`** | **0.30..0.50** | **5** ← *co-evolui* |

### Fitness com penalidade suave

```python
return f2 - ALPHA * max(0.0, 0.80 - recall) ** 2
```

A descontinuidade (cliff) do piso 0.80 é substituída por um gradiente contínuo.
Indivíduos "quase bons" (recall=0.79) têm fitness ~f2 e podem cruzar com bons;
a seleção direcional funciona em todo o espaço em vez de dividir em "mortos/vivos".

In [ ]:
_cv_strategy = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
SEARCH_SPACE: dict = {
    "max_iter":          [100, 150, 200, 250, 300, 350, 400, 500],
    "learning_rate":     [0.01, 0.03, 0.05, 0.08, 0.1, 0.15, 0.2],
    "max_leaf_nodes":    [15, 23, 31, 47, 63, 95, 127],
    "min_samples_leaf":  [10, 20, 35, 50, 75, 100, 150],
    "l2_regularization": [0.0, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0],
    "max_depth":         [None, 3, 5, 7, 10, 15],
    "max_bins":          [63, 127, 255],
}
THRESHOLD_VALUES = [0.30, 0.35, 0.40, 0.45, 0.50]

PARAM_NAMES      = list(SEARCH_SPACE.keys())
PARAM_VALUES     = list(SEARCH_SPACE.values())
N_HPARAM_GENES   = len(PARAM_NAMES)
N_THRESHOLD_VALS = len(THRESHOLD_VALUES)
N_GENES          = N_HPARAM_GENES + 1   # último gene = threshold

# Unifica limites de todos os genes (hiper + threshold)
GENE_SIZES = [len(v) for v in PARAM_VALUES] + [N_THRESHOLD_VALS]

def decode(individual: np.ndarray) -> tuple:
    """Retorna (params_histgb, threshold)."""
    params    = {PARAM_NAMES[i]: PARAM_VALUES[i][individual[i]] for i in range(N_HPARAM_GENES)}
    threshold = THRESHOLD_VALUES[individual[N_HPARAM_GENES]]
    return params, threshold

def random_individual(rng: np.random.Generator) -> np.ndarray:
    return np.array([rng.integers(0, size) for size in GENE_SIZES])

# Baseline: hiper idênticos ao 03 + threshold=0.40 (índice 2 em THRESHOLD_VALUES)
# max_iter=350→5, lr=0.05→2, max_leaf_nodes=31→2, min_samples_leaf=100→5
# l2=1.0→4, max_depth=None→0, max_bins=255→2, threshold=0.40→2
_baseline_individual = np.array([5, 2, 2, 5, 4, 0, 2, 2])
_bl_params, _bl_thr  = decode(_baseline_individual)

total_comb = 1
for s in GENE_SIZES:
    total_comb *= s

print(f"Genes total : {N_GENES}  (7 hiperparâmetros + 1 threshold)")
print(f"Combinações : {total_comb:,}")
print(f"Baseline    : {_bl_params}  |  threshold={_bl_thr}")
assert _baseline_individual[N_HPARAM_GENES] == 2, "threshold gene index errado"
assert _bl_thr == 0.40, f"threshold esperado 0.40, obtido {_bl_thr}"
print("Asserts OK")

In [ ]:
def fitness(individual: np.ndarray) -> float:
    """
    Fitness co-evolutivo com penalidade quadrática suave.
    - Decoda hiperparâmetros E threshold do mesmo cromossomo.
    - Scorer é construído dinamicamente com o threshold do indivíduo.
    - Sem cliff: pressão seletiva contínua em todo o espaço.
    """
    params, threshold = decode(individual)

    def _recall_fn(y_true, y_prob, **kwargs):
        return recall_score(y_true, (y_prob >= threshold).astype(int), zero_division=0)

    def _f2_fn(y_true, y_prob, **kwargs):
        return fbeta_score(y_true, (y_prob >= threshold).astype(int), beta=2, zero_division=0)

    recall_scorer = make_scorer(_recall_fn, needs_proba=True)
    f2_scorer     = make_scorer(_f2_fn,     needs_proba=True)

    model = HistGradientBoostingClassifier(**params, random_state=RANDOM_STATE)

    mean_recall = cross_val_score(
        model, X_train, y_train, cv=_cv_strategy, scoring=recall_scorer, n_jobs=1,
    ).mean()
    mean_f2 = cross_val_score(
        model, X_train, y_train, cv=_cv_strategy, scoring=f2_scorer, n_jobs=1,
    ).mean()

    # Penalidade quadrática suave — zero quando recall >= 0.80
    penalty = ALPHA * max(0.0, MIN_RECALL_THRESHOLD - float(mean_recall)) ** 2
    return float(mean_f2) - penalty


# Verificação baseline
_bl_cv_fitness = fitness(_baseline_individual)
_bl_test       = evaluate_on_test(_bl_params, _bl_thr)
print(f"Baseline individual : {_baseline_individual}")
print(f"Threshold decodificado : {_bl_thr}")
print(f"Fitness CV (suave)  : {_bl_cv_fitness:.4f}")
print(f"Recall no teste (thr={_bl_thr}) : {_bl_test['recall']:.4f}")
print(f"F2     no teste (thr={_bl_thr}) : {_bl_test['f2']:.4f}")

## Operadores Genéticos

Idênticos aos notebooks anteriores — torneio, crossover de ponto único, mutação por gene.
A única diferença é que `mutate` e `random_individual` usam `GENE_SIZES`
para respeitar os limites de cada gene (inclusive o threshold).

In [ ]:
def tournament_selection(population, fitnesses, tournament_size, rng):
    candidates   = rng.choice(len(population), size=tournament_size, replace=False)
    winner_index = candidates[np.argmax([fitnesses[i] for i in candidates])]
    return population[winner_index].copy()

def single_point_crossover(parent_a, parent_b, crossover_rate, rng):
    if rng.random() > crossover_rate:
        return parent_a.copy(), parent_b.copy()
    cut_point = rng.integers(1, N_GENES)
    child_a = np.concatenate([parent_a[:cut_point], parent_b[cut_point:]])
    child_b = np.concatenate([parent_b[:cut_point], parent_a[cut_point:]])
    return child_a, child_b

def mutate(individual, mutation_rate, rng):
    mutated = individual.copy()
    for gene_index in range(N_GENES):
        if rng.random() < mutation_rate:
            mutated[gene_index] = rng.integers(0, GENE_SIZES[gene_index])
    return mutated

## Loop Principal — AG com Mutação Adaptativa

Quando `best_fitness` não melhora por `STAGNATION_WINDOW` gerações consecutivas,
a taxa de mutação é multiplicada por `MUTATION_BOOST` até o próximo progresso.

Biologicamente: **mutagênese induzida por estresse** — organismos sob pressão
elevam a taxa de mutação para escapar de ótimos locais.

In [ ]:
def _initialize_population(pop_size: int, rng: np.random.Generator, seed_individual=None) -> list:
    pop = [seed_individual.copy()] if seed_individual is not None else []
    pop += [random_individual(rng) for _ in range(pop_size - len(pop))]
    return pop

def _evaluate_population(population: list) -> list:
    return Parallel(n_jobs=N_JOBS)(
        delayed(fitness)(individual) for individual in population
    )

def _breed_next_generation(
    population, fitnesses, pop_size, elite_size,
    tournament_size, crossover_rate, mutation_rate, rng,
):
    sorted_by_fitness = np.argsort(fitnesses)[::-1]
    next_generation   = [population[i].copy() for i in sorted_by_fitness[:elite_size]]

    while len(next_generation) < pop_size:
        parent_a = tournament_selection(population, fitnesses, tournament_size, rng)
        parent_b = tournament_selection(population, fitnesses, tournament_size, rng)
        child_a, child_b = single_point_crossover(parent_a, parent_b, crossover_rate, rng)
        next_generation.append(mutate(child_a, mutation_rate, rng))
        if len(next_generation) < pop_size:
            next_generation.append(mutate(child_b, mutation_rate, rng))

    return next_generation

def run_genetic_algorithm(
    pop_size: int         = 10,
    n_generations: int    = 8,
    mutation_rate: float  = 0.20,
    crossover_rate: float = 0.80,
    tournament_size: int  = 3,
    elite_size: int       = 1,
    seed: int             = RANDOM_STATE,
    label: str            = "AG",
    seed_individual       = None,
) -> dict:
    rng        = np.random.default_rng(seed)
    population = _initialize_population(pop_size, rng, seed_individual)

    best_individual       = None
    best_fitness_ever     = -np.inf
    history_rows          = []
    stagnation_count      = 0
    current_mutation_rate = mutation_rate

    gen_width = len(str(n_generations))

    for generation in range(1, n_generations + 1):
        fitnesses = _evaluate_population(population)

        gen_best_idx     = int(np.argmax(fitnesses))
        gen_best_fitness = fitnesses[gen_best_idx]
        gen_best_ind     = population[gen_best_idx]
        gen_best_params, gen_best_thr = decode(gen_best_ind)

        if gen_best_fitness > best_fitness_ever + 1e-6:
            best_fitness_ever     = gen_best_fitness
            best_individual       = gen_best_ind.copy()
            stagnation_count      = 0
            current_mutation_rate = mutation_rate
        else:
            stagnation_count += 1
            if stagnation_count >= STAGNATION_WINDOW:
                current_mutation_rate = min(mutation_rate * MUTATION_BOOST, 0.80)

        history_rows.append({
            "generation":         generation,
            "best_fitness":       gen_best_fitness,
            "mean_fitness":       float(np.mean(fitnesses)),
            "best_threshold":     gen_best_thr,
            "mutation_rate_used": current_mutation_rate,
            "stagnation_count":   stagnation_count,
        })

        mut_tag = f" ⚡mut→{current_mutation_rate:.2f}" if current_mutation_rate > mutation_rate else ""
        print(
            f"[{label}] Ger {generation:{gen_width}}/{n_generations} | "
            f"melhor={gen_best_fitness:.4f} | média={np.mean(fitnesses):.4f} | "
            f"thr={gen_best_thr:.2f}{mut_tag}"
        )

        population = _breed_next_generation(
            population, fitnesses, pop_size, elite_size,
            tournament_size, crossover_rate, current_mutation_rate, rng,
        )

    best_params, best_threshold = decode(best_individual)
    return {
        "label":           label,
        "best_individual": best_individual,
        "best_params":     best_params,
        "best_threshold":  best_threshold,
        "best_cv_fitness": best_fitness_ever,
        "history":         pd.DataFrame(history_rows),
    }

In [ ]:
_smoke = run_genetic_algorithm(pop_size=4, n_generations=2, label="smoke_06", seed=RANDOM_STATE)

print(f"\nSmoke OK")
print(f"Params    : {_smoke['best_params']}")
print(f"Threshold : {_smoke['best_threshold']}")
print(f"Fitness   : {_smoke['best_cv_fitness']:.4f}")

_sp, _st = decode(_smoke["best_individual"])
assert all(v in PARAM_VALUES[i] for i, v in enumerate(_sp.values())), "hparam fora do espaço"
assert _st in THRESHOLD_VALUES, "threshold fora do espaço"
print("Validação do espaço: OK")

## Experimentos — Cenário B | Co-evolução + Penalidade Suave

Quatro configurações do AG (mesmos rótulos dos notebooks anteriores para comparabilidade),
agora com o genoma co-evolutivo de 8 genes e fitness suave.

In [ ]:
EXPERIMENT_CONFIGS = [
    dict(pop_size=10, n_generations=8, mutation_rate=0.10, crossover_rate=0.70, label="Exp1_conservador"),
    dict(pop_size=10, n_generations=8, mutation_rate=0.20, crossover_rate=0.80, label="Exp2_padrao"),
    dict(pop_size=10, n_generations=8, mutation_rate=0.40, crossover_rate=0.80, label="Exp3_exploratorio"),
    dict(pop_size=10, n_generations=8, mutation_rate=0.20, crossover_rate=0.80,
         label="Exp4_hotstart", seed_individual=_baseline_individual),
]

In [ ]:
experiment_results_B: dict = {}

print(f"{'='*65}")
print("Cenário B | Co-evolução threshold + hiperparâmetros | penalidade suave")
print(f"{'='*65}")

for config in EXPERIMENT_CONFIGS:
    print(f"\n{'─'*65}\n{config['label']}\n{'─'*65}")
    result = run_genetic_algorithm(**config, seed=RANDOM_STATE)
    experiment_results_B[config["label"]] = result
    print(f"  Threshold co-evoluído : {result['best_threshold']}")
    print(f"  Melhores params       : {result['best_params']}")
    print(f"  Fitness CV            : {result['best_cv_fitness']:.4f}")

gc.collect()

In [ ]:
fig, axes = plt.subplots(1, len(EXPERIMENT_CONFIGS), figsize=(5 * len(EXPERIMENT_CONFIGS), 4), sharey=True)

for ax, config in zip(axes, EXPERIMENT_CONFIGS):
    label = config["label"]
    hist  = experiment_results_B[label]["history"]
    ax.plot(hist["generation"], hist["best_fitness"],  "o-",  label="melhor")
    ax.plot(hist["generation"], hist["mean_fitness"],  "s--", label="média", alpha=0.7)
    # Marca gerações com mutação amplificada
    boosted = hist[hist["mutation_rate_used"] > config["mutation_rate"]]
    if not boosted.empty:
        ax.scatter(boosted["generation"], boosted["best_fitness"],
                   marker="*", s=120, color="red", zorder=5, label="mut. amplificada")
    ax.set_title(label.replace("_", " "), fontsize=9)
    ax.set_xlabel("Geração")
    ax.set_ylabel("Fitness")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle("Convergência — Co-evolução | Penalidade Suave | Cenário B", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_convergence_coevo_B.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(EXPERIMENT_CONFIGS), figsize=(5 * len(EXPERIMENT_CONFIGS), 3), sharey=True)

for ax, config in zip(axes, EXPERIMENT_CONFIGS):
    label = config["label"]
    hist  = experiment_results_B[label]["history"]
    ax.plot(hist["generation"], hist["best_threshold"], "D-", color="darkorange", linewidth=2)
    ax.set_yticks(THRESHOLD_VALUES)
    ax.set_title(label.replace("_", " "), fontsize=9)
    ax.set_xlabel("Geração")
    ax.set_ylabel("Threshold co-evoluído")
    ax.grid(True, alpha=0.3)

plt.suptitle("Evolução do Threshold ao Longo das Gerações | Cenário B", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_threshold_evolution_B.png", dpi=120, bbox_inches="tight")
plt.show()

## Cenários A / B / C — Teste de Generalização

O melhor config do AG (Exp2_padrao) é rodado nos três conjuntos de features para
verificar se o AG co-evolutivo se comporta diferente com outras representações dos dados.

Biologicamente: **especiação por cenário** — cada subpopulação evolui no seu nicho.

In [ ]:
_scenario_config = dict(pop_size=10, n_generations=8, mutation_rate=0.20, crossover_rate=0.80)

scenario_results: dict = {}

for scenario in ["A", "B", "C"]:
    print(f"\n{'='*65}")
    print(f"Cenário {scenario} — {SCENARIO_DESCRIPTIONS[scenario]}")
    print(f"{'='*65}")

    X_train, X_test, y_train, y_test = load_scenario_data(scenario)
    print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")

    result = run_genetic_algorithm(
        **_scenario_config,
        label=f"Cenario_{scenario}",
        seed=RANDOM_STATE,
    )
    scenario_results[scenario] = result

    print(f"\n  Threshold co-evoluído : {result['best_threshold']}")
    print(f"  Melhores params       : {result['best_params']}")
    print(f"  Fitness CV            : {result['best_cv_fitness']:.4f}")

# Restaura cenário B como padrão
X_train, X_test, y_train, y_test = load_scenario_data("B")
print("\nCenário B restaurado.")
gc.collect()

## Comparativo Final

Avaliação no conjunto de teste para todos os modelos:
- Baseline com threshold 0.40 e 0.50
- Melhor resultado AG em cenário B
- AG rodado em cada cenário (A/B/C)

In [ ]:
def build_row(label, params, threshold, cv_fitness, scenario):
    X_tr, X_te, y_tr, y_te = load_scenario_data(scenario)
    sw    = compute_sample_weight(class_weight="balanced", y=y_tr)
    model = HistGradientBoostingClassifier(**params, random_state=RANDOM_STATE)
    model.fit(X_tr, y_tr, sample_weight=sw)
    y_prob = model.predict_proba(X_te)[:, 1]
    metrics = compute_metrics(y_te, y_prob, threshold=threshold)
    return {
        "origem":     label,
        "cenario":    scenario,
        "threshold":  threshold,
        "cv_fitness": round(cv_fitness, 4),
        **{m: metrics[m] for m in METRIC_NAMES},
    }

rows = [
    build_row("Baseline (Fase 1) thr=0.50", BASELINE_PARAMS, 0.50, _bl_cv_fitness, "B"),
    build_row("Baseline (Fase 1) thr=0.40", BASELINE_PARAMS, 0.40, _bl_cv_fitness, "B"),
]

# Melhor resultado AG cenário B
best_exp_label = max(experiment_results_B, key=lambda k: experiment_results_B[k]["best_cv_fitness"])
best_B = experiment_results_B[best_exp_label]
rows.append(build_row(
    f"AG Coevo | {best_exp_label}", best_B["best_params"], best_B["best_threshold"],
    best_B["best_cv_fitness"], "B",
))

# Todos os cenários
for scenario, result in scenario_results.items():
    rows.append(build_row(
        f"AG Coevo | Cenário {scenario}", result["best_params"], result["best_threshold"],
        result["best_cv_fitness"], scenario,
    ))

comparison_df = pd.DataFrame(rows).set_index("origem")
print("=== Comparativo Final ===")
display(comparison_df.round(4))

comparison_df.to_csv(METRICS_DIR / "06_ag_coevo_comparison.csv")
print(f"\nSalvo: {METRICS_DIR / '06_ag_coevo_comparison.csv'}")
gc.collect()

In [ ]:
PIVOT_THRESHOLD_FAIL    = 0.01
PIVOT_THRESHOLD_SUCCESS = 0.03
MAX_FP_GROWTH_PCT       = 0.15
MAX_FN_GROWTH_PCT       = 0.30
MIN_RECALL_TEST         = 0.80

print("=" * 65)
print("ANÁLISE DE PIVÔ — Co-evolução vs Baseline (Cenário B)")
print("=" * 65)

baseline_row = comparison_df.loc["Baseline (Fase 1) thr=0.40"]
ag_row_label = f"AG Coevo | {best_exp_label}"
ag_row       = comparison_df.loc[ag_row_label]

delta_f2        = ag_row["f2"]        - baseline_row["f2"]
delta_recall    = ag_row["recall"]    - baseline_row["recall"]
delta_precision = ag_row["precision"] - baseline_row["precision"]
thr_coevo       = ag_row["threshold"]

print(f"\nThreshold co-evoluído pelo AG : {thr_coevo} (baseline fixo era 0.40)")
print(f"F2       : {ag_row['f2']:.4f}   (baseline: {baseline_row['f2']:.4f}   | delta: {delta_f2:+.4f})")
print(f"Recall   : {ag_row['recall']:.4f}   (baseline: {baseline_row['recall']:.4f}   | delta: {delta_recall:+.4f})")
print(f"Precisão : {ag_row['precision']:.4f}   (baseline: {baseline_row['precision']:.4f}   | delta: {delta_precision:+.4f})")

n_prematuros = int(y_test.sum())
fn_base = int(n_prematuros * (1 - baseline_row["recall"]))
fn_ag   = int(n_prematuros * (1 - ag_row["recall"]))
tp_base = n_prematuros - fn_base
tp_ag   = n_prematuros - fn_ag
fp_base = max(0, int(tp_base / max(baseline_row["precision"], 1e-9) - tp_base))
fp_ag   = max(0, int(tp_ag   / max(ag_row["precision"],   1e-9) - tp_ag))
fp_growth = (fp_ag - fp_base) / max(fp_base, 1)
fn_growth = (fn_ag - fn_base) / max(fn_base, 1)

print(f"\n{'':35} {'Baseline':>10} {'AG Coevo':>10} {'Delta':>10}")
print(f"{'FN (prematuros perdidos)':35} {fn_base:>10,} {fn_ag:>10,} {fn_ag-fn_base:>+10,}")
print(f"{'FP (alarmes falsos)':35} {fp_base:>10,} {fp_ag:>10,} {fp_ag-fp_base:>+10,}")

recall_ok = ag_row["recall"] >= MIN_RECALL_TEST
fn_ok     = fn_growth <= MAX_FN_GROWTH_PCT
fp_ok     = fp_growth <= MAX_FP_GROWTH_PCT

print(f"\n--- Validação clínica ---")
print(f"  Recall >= {MIN_RECALL_TEST}: {'✓' if recall_ok else '✗'}  {ag_row['recall']:.4f}")
print(f"  FN delta <= {MAX_FN_GROWTH_PCT:.0%}  : {'✓' if fn_ok else '✗'}  {fn_growth:+.1%}")
print(f"  FP delta <= {MAX_FP_GROWTH_PCT:.0%}  : {'✓' if fp_ok else '✗'}  {fp_growth:+.1%}")

print(f"\n--- Critério de pivô ---")
if delta_f2 >= PIVOT_THRESHOLD_SUCCESS and recall_ok and fn_ok:
    print(f"✓ Melhoria expressiva ({delta_f2:+.4f}) — AG co-evolutivo bem-sucedido.")
    print(f"  Threshold ótimo encontrado: {thr_coevo} (vs 0.40 assumido anteriormente)")
elif delta_f2 >= PIVOT_THRESHOLD_FAIL:
    print(f"~ Melhoria marginal ({delta_f2:+.4f}) — co-evolução moveu threshold mas ganho em F2 é pequeno.")
    print(f"  Threshold ótimo encontrado: {thr_coevo}")
    print("  Próximo passo: NSGA-II multi-objetivo para explorar fronteira recall×precisão.")
else:
    print(f"✗ Sem melhoria ({delta_f2:+.4f}) — dados são o teto do modelo, não os hiperparâmetros.")
    print(f"  Threshold co-evoluído: {thr_coevo}")
    print("  Recomendação: threshold estratificado por perfil de risco ou feature engineering.")

print("\n--- Cenários A/B/C ---")
for scenario, result in scenario_results.items():
    s_row = comparison_df.loc[f"AG Coevo | Cenário {scenario}"]
    print(f"  Cenário {scenario} ({SCENARIO_DESCRIPTIONS[scenario]:20}) | "
          f"threshold={result['best_threshold']} | "
          f"recall={s_row['recall']:.4f} | f2={s_row['f2']:.4f}")